# 00 — Setup & Environment Check

Day 1 deliverable. Run this on a **Colab or Kaggle T4 GPU runtime**.

Confirms:
- GPU is available and reports expected VRAM (T4 = 16GB)
- Dependencies install cleanly
- The retriever base model (`all-MiniLM-L6-v2`) loads
- The generator (`Qwen2.5-1.5B-Instruct`) loads in fp16, has 28 layers (this
  is why it was chosen over Llama-3.2-1B — see `docs/DECISIONS.md` §1), and a
  forward pass returns hidden states of the expected shape.

**Refresher (transformer residual stream):** each transformer layer reads
from and writes back to a running sum called the *residual stream*. With
`output_hidden_states=True`, Hugging Face returns a tuple of length
`num_hidden_layers + 1`: index 0 is the input embeddings (before any layer
runs), and index `i` (for `i >= 1`) is the residual stream state *after*
layer `i` has added its attention + FFN output. Each tensor has shape
`[batch, seq_len, d_model]`.

## 1. Mount Drive (Colab only)

Colab's local disk is wiped on every runtime restart, so `checkpoints/`,
`data/synthetic/`, and `data/probing/` need to live on Drive to persist
across sessions. Kaggle instead persists via "Save Version" + exported
Datasets — no Drive mount needed there.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "kaggle_secrets" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in __import__("os").environ

print(f"IN_COLAB={IN_COLAB}, IN_KAGGLE={IN_KAGGLE}")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/temporal-conflict-rag"
    import os
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f"Drive artifact root: {DRIVE_ROOT}")
else:
    DRIVE_ROOT = None
    print("Not on Colab — skipping Drive mount.")

## 2. Clone the project repo

Once the repo has a GitHub remote, each session starts by cloning/pulling
here instead of re-uploading files by hand. Fill in `REPO_URL` once it
exists; harmless to skip for now if you're running from an already-synced
local clone.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<user>/temporal-conflict-rag.git"

if REPO_URL and (IN_COLAB or IN_KAGGLE):
    import subprocess
    subprocess.run(["git", "clone", REPO_URL], check=False)
    print("Cloned. cd into the repo directory before running later notebooks.")
else:
    print("REPO_URL not set (or running locally) — skipping clone.")

## 3. Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## 4. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — switch the Colab/Kaggle runtime to a GPU (T4)."

device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print(f"Device: {props.name}")
print(f"Total VRAM: {props.total_memory / 1e9:.2f} GB")
print(f"torch version: {torch.__version__}, CUDA: {torch.version.cuda}")

## 5. Load the retriever base model

`sentence-transformers` wraps `AutoModel` with mean pooling + L2 normalization
baked in — different from calling `AutoModel` directly and pooling yourself.
This is the *frozen baseline* we'll fine-tune against in Stage 1 (Day 3–4).

In [ ]:
from sentence_transformers import SentenceTransformer

retriever = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
n_params = sum(p.numel() for p in retriever.parameters())
print(f"Retriever params: {n_params / 1e6:.1f}M")

# Smoke test: embed a couple of strings, confirm shape + normalization
emb = retriever.encode(["policy updated 2026", "policy updated 2019"], convert_to_tensor=True)
print(f"Embedding shape: {tuple(emb.shape)}")
print(f"Norms (should be ~1.0): {emb.norm(dim=1)}")

## 6. Load the generator (Qwen2.5-1.5B-Instruct, fp16)

Per `docs/DECISIONS.md`: fp16, not 4-bit — quantization noise would corrupt
the hidden states the Stage 3 probe depends on. This cell confirms the layer
count (must be 28, matching the proposal's "layers 18–24" probing target) and
reports VRAM used after load.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
generator = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_ID, torch_dtype=torch.float16
).to("cuda")
generator.eval()

print(f"num_hidden_layers: {generator.config.num_hidden_layers} (expect 28)")
print(f"hidden_size (d_model): {generator.config.hidden_size}")
print(f"VRAM allocated after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 7. Forward-pass smoke test: extract hidden states

This is the exact extraction path Stage 3 will use (`output_hidden_states=True`
on a plain forward pass) — confirmed here before any pipeline code depends on
it. Expect `len(hidden_states) == 29` (28 layers + the input embedding layer).

In [ ]:
prompt = "Context: The subscription price increased to $49/month in March 2026.\nQuery: What is the current subscription price?\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = generator(**inputs, output_hidden_states=True)

hidden_states = out.hidden_states
print(f"Number of hidden_states tensors: {len(hidden_states)} (expect 29 = 28 layers + embeddings)")
print(f"Shape of each: {tuple(hidden_states[0].shape)}  [batch, seq_len, d_model]")
print(f"Final-token hidden state at layer 20 (mid-range of the 18-24 probe target): {hidden_states[20][0, -1, :5]}")

## 8. Summary

If every cell above ran without error, the environment is ready for Day 2
(synthetic data generation) and beyond. Record the printed VRAM numbers here
for reference against the proposal's estimates (~3.2GB retriever fine-tuning,
~4.5GB generator inference/probing).

In [ ]:
print("Environment check complete.")
print(f"Peak VRAM so far: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")